In [1]:
!uv pip install --quiet tokenizers datasets torch

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from datasets import load_dataset, disable_progress_bar
from typing import Iterator

disable_progress_bar()

## BPETokenizer (from notebook 02)

In [3]:
class BPETokenizer:
    SPECIAL_TOKENS = ["[UNK]", "[PAD]", "[BOS]", "[EOS]"]

    def __init__(self, vocab_size: int = 32768):
        self.vocab_size = vocab_size
        self._tokenizer: Tokenizer | None = None

    def _batch_iterator(self, dataset, text_column: str = "text", batch_size: int = 1000) -> Iterator[list[str]]:
        for i in range(0, len(dataset), batch_size):
            yield dataset[i : i + batch_size][text_column]

    def train(self, dataset, text_column: str = "text") -> None:
        tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=self.SPECIAL_TOKENS,
            min_frequency=2,
            show_progress=True,
        )
        tokenizer.train_from_iterator(
            self._batch_iterator(dataset, text_column),
            trainer=trainer,
            length=len(dataset),
        )
        bos_id = tokenizer.token_to_id("[BOS]")
        eos_id = tokenizer.token_to_id("[EOS]")
        tokenizer.post_processor = TemplateProcessing(
            single="[BOS] $A [EOS]",
            special_tokens=[("[BOS]", bos_id), ("[EOS]", eos_id)],
        )
        self._tokenizer = tokenizer

    def encode(self, text: str) -> list[int]:
        return self._tokenizer.encode(text).ids

    def encode_batch(self, texts: list[str]) -> list[list[int]]:
        return [enc.ids for enc in self._tokenizer.encode_batch(texts)]

    def decode(self, ids: list[int]) -> str:
        return self._tokenizer.decode(ids)

    def save(self, path: str) -> None:
        self._tokenizer.save(path)

    @classmethod
    def load(cls, path: str) -> "BPETokenizer":
        obj = cls()
        obj._tokenizer = Tokenizer.from_file(path)
        obj.vocab_size = obj._tokenizer.get_vocab_size()
        return obj

    @property
    def pad_id(self) -> int:
        return self._tokenizer.token_to_id("[PAD]")

    @property
    def vocab_size_actual(self) -> int:
        return self._tokenizer.get_vocab_size()

    def __repr__(self) -> str:
        trained = self._tokenizer is not None
        size = self.vocab_size_actual if trained else "?"
        return f"BPETokenizer(vocab_size={size}, trained={trained})"

## CausalLMDataset

In [4]:
class CausalLMDataset(Dataset):
    """
    Tokenizes a HuggingFace dataset and chunks it into fixed-length windows.

    Each item returns:
        input_ids  — token ids [0 .. seq_len-1]
        labels     — token ids [1 .. seq_len]   (next-token targets)

    Documents are concatenated with EOS between them so context never
    crosses a padding boundary.
    """

    def __init__(self, hf_dataset, tokenizer: BPETokenizer, seq_len: int = 512, text_column: str = "text"):
        self.seq_len = seq_len
        self.tokenizer = tokenizer

        # Flatten all documents into one long token stream
        token_stream: list[int] = []
        eos_id = tokenizer._tokenizer.token_to_id("[EOS]")
        for example in hf_dataset:
            ids = tokenizer.encode(example[text_column])
            token_stream.extend(ids)
            token_stream.append(eos_id)

        # Discard the trailing partial window
        total = len(token_stream)
        n_chunks = total // (seq_len + 1)
        token_stream = token_stream[: n_chunks * (seq_len + 1)]

        # Shape: (n_chunks, seq_len + 1)
        self._tokens = torch.tensor(token_stream, dtype=torch.long).view(n_chunks, seq_len + 1)

    def __len__(self) -> int:
        return len(self._tokens)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        chunk = self._tokens[idx]
        return {
            "input_ids": chunk[:-1].clone(),
            "labels":    chunk[1:].clone(),
        }

## Usage

In [5]:
# Load dataset (same slice used in notebook 02)
hf_dataset = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=False)
hf_dataset = hf_dataset.select(range(1000))

# Train tokenizer
tok = BPETokenizer(vocab_size=32768)
tok.train(hf_dataset)
print(tok)




BPETokenizer(vocab_size=32699, trained=True)


In [6]:
SEQ_LEN = 512
BATCH_SIZE = 32

ds = CausalLMDataset(hf_dataset, tokenizer=tok, seq_len=SEQ_LEN)
print(f"Chunks : {len(ds):,}")
print(f"Tokens : {len(ds) * SEQ_LEN:,}")

Chunks : 1,252
Tokens : 641,024


In [10]:
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True if torch.cuda.is_available() else False)

# Inspect one batch
batch = next(iter(loader))
print("input_ids :", batch["input_ids"].shape)   # (32, 512)
print("labels    :", batch["labels"].shape)       # (32, 512)
print("sample    :", tok.decode(batch["input_ids"][0].tolist())[:120])

input_ids : torch.Size([32, 512])
labels    : torch.Size([32, 512])
sample    : as deal with a worry wart mom , a homework - eating sn ake , and a coach with an ear wax problem . If you laugh - sn ort
